# 04. 응급도 분류 모델 실험 보고서 (v0)

backlog `04-acuity-model.md`에서 한 실험을 보고서 형식으로 정리한다. 다루는 것은 네 가지다.

1. 어떤 데이터셋을 썼는가
2. 데이터를 실제로 어떻게 탐색했는가
3. 어떻게 전처리했는가
4. 하이퍼파라미터와 임계값을 어떻게 정했는가

그 뒤에 결과(검증셋·교차검증·최종 test), SHAP 판단 근거, 한계를 적는다.

**읽는 법.** 각 절은 **[실제 진행]**과 **[사후 확인]**을 나눠 적는다.

- **[실제 진행]**: 04를 하면서 실제로 본 것과 정한 것이다. 모델은 이 기준으로 만들어졌다.
- **[사후 확인]**: 이 보고서를 쓰면서 train+valid로 새로 계산한 표와 그림이다. 모델과 임계값을 바꾸는 데 쓰지 않았다.

**test 데이터 규칙.** 이 노트북은 `data/test/`를 읽지 않는다. 탐색은 train+valid 1,013행만 쓴다. test 수치는 평가 명령(`scripts/evaluate.py`)이 한 번 남긴 `reports/acuity_v0_test.json`을 읽어 옮긴다.

**실행.** `cd modeling && uv sync --group notebook` 후, 이 프로젝트 `.venv` 커널로 위에서부터 실행한다. 몇 초면 끝난다. 커밋할 때는 출력을 지운다(`modeling/CLAUDE.md`).

## 0. 요약

| 항목 | 값 |
|---|---|
| 데이터 | Moon et al. 2019 PLOS ONE S1 Appendix(국내 응급실 2곳, 1,267건, CC BY 4.0). Kaggle "Emergency Service – Triage Application"의 원 출처 |
| 라벨 | 전문가 KTAS → 3단계: 1–2 `emergency`, 3 `today`, 4–5 `outpatient` |
| 피처 | 버튼으로 받을 수 있는 34개(부위 13, 증상 15, 통증 점수, 의식 AVPU, 나이대, 발생 시점, 성별, 손상 여부). 활력징후는 쓰지 않음 |
| 분할 | 층화 train 823 / valid 190 / test 254(65 / 15 / 20%), id 목록으로 고정 |
| 모델 | LightGBM 다중 분류(서빙 후보), 비교용 다항 로지스틱 회귀 |
| 하이퍼파라미터 | 작은 데이터용 보수적인 값을 탐색 없이 한 번 정함. 부스팅 라운드 수(86)만 train 안 5겹 교차검증 조기 종료로 정함 |
| 임계값 | valid 응급 재현율 95% 이상 중 정밀도 최대 → `emergency` 확률 ≥ 0.0738이면 응급 |
| valid | 응급 재현율 36/37, 응급 정밀도 36/99 |
| train+valid 5겹 CV | 응급 재현율 184/197(93.4%), 응급 정밀도 184/500 |
| **최종 test(1회)** | **응급 재현율 42/49(85.7%)**, 응급 정밀도 42/129, 정확도 140/254 |

test 응급 재현율이 valid에서 정한 목표 95%에 못 미친다. 응급이 아닌 사람도 많이 응급으로 분류한다(test에서 응급 판정 129건 중 87건은 응급이 아님).

In [ ]:
import json
import sys
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager
from matplotlib.ticker import FuncFormatter
from sklearn.model_selection import StratifiedKFold

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "src" / "fornav_modeling").exists())
sys.path.insert(0, str(ROOT / "src"))  # package = false라 설치되지 않으므로 src를 직접 찾는다

from fornav_modeling import explain  # noqa: E402
from fornav_modeling.data import split  # noqa: E402
from fornav_modeling.features.buttons import FEATURE_NAMES, featurize_many  # noqa: E402
from fornav_modeling.labels.ktas import ACUITIES  # noqa: E402
from fornav_modeling.models import acuity  # noqa: E402
from fornav_modeling.models.acuity import EMERGENCY, LGBM_PARAMS, LightGBMAcuity, encode_labels  # noqa: E402

DATA = ROOT / "data"
REPORTS = ROOT / "reports"
MODEL_DIR = ROOT / "models" / "acuity" / "v0"

# 그림: 밝은 바탕, 가는 선, 옅은 격자. 색은 파랑·주황 두 가지(색각 이상 검사 통과)
BLUE, ORANGE, INK, MUTED, GRID, SURFACE = "#2a78d6", "#eb6834", "#1f1f1e", "#6b6a64", "#e4e3de", "#fcfcfb"
_installed = {f.name for f in font_manager.fontManager.ttflist}
KOREAN_FONT = next(
    (f for f in ("Apple SD Gothic Neo", "NanumGothic", "Malgun Gothic", "Noto Sans CJK KR") if f in _installed), "DejaVu Sans"
)
plt.rcParams.update(
    {
        "font.family": KOREAN_FONT,
        "axes.unicode_minus": False,
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": GRID,
        "axes.labelcolor": INK,
        "text.color": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "lines.linewidth": 2,
        "figure.dpi": 110,
    }
)
pd.set_option("display.max_rows", 60)

splits_report = json.loads((REPORTS / "triage_splits.json").read_text())
valid_report = json.loads((REPORTS / "acuity_v0.json").read_text())
test_report = json.loads((REPORTS / "acuity_v0_test.json").read_text())
metadata = json.loads((MODEL_DIR / "metadata.json").read_text())

## 1. 데이터셋

### 1.1 출처와 사용 조건 [실제 진행: 02]

| 항목 | 내용 |
|---|---|
| 논문 | Moon S-H, Shim JL, Park K-S, Park C-S (2019). Triage accuracy and causes of mistriage using the Korean Triage and Acuity Scale. *PLoS ONE* 14(9): e0216972 |
| 파일 | S1 Appendix "EMR review coding data", `data/raw/kaggle_triage/pone.0216972.s001.xlsx` |
| sha256 | `f631c951178799179d0fbd74fe979464b955ea1b426222bf015b163b8a24beb0` |
| 라이선스 | CC BY 4.0. 출처 표시 조건으로 학습·평가·상업 이용 가능 |
| 규모 | 1,267행 × 24열. 국내 응급실 2곳(지역응급의료기관, 권역응급의료센터)의 EMR 검토 기록 |
| 식별 정보 | 없음(이름·연락처·병원명 열 없음, 자유 텍스트에서 이메일·전화번호 형태 없음) |

README에는 데이터 출처가 Kaggle "Emergency Service – Triage Application"으로 적혀 있다. 하지만 Kaggle 판은 라이선스 문구가 없다. 그래서 같은 내용의 원 출처인 PLOS ONE 부록을 받았다(02에서 사용자와 정함). 내려받기와 해시 검증은 `scripts/download_data.py`가 한다. 자세한 내용은 `data/README.md`에 있다.

### 1.2 원본 열 [실제 진행: 02]

원본 24열은 쓰임에 따라 네 가지로 나뉜다.

| 쓰임 | 열 |
|---|---|
| 버튼 피처로 바꿔 씀 | `Sex`, `Age`, `Injury`, `Chief_complain`(주증상 자유 텍스트), `Mental`(AVPU 1–4), `Pain`(있음/없음), `NRS_pain`(0–10) |
| 라벨 | `KTAS_expert`(전문가 KTAS, 라벨), `KTAS_RN`(간호사 KTAS, 비교용) |
| 09 비교 실험 전용 | `SBP`, `DBP`, `HR`, `RR`, `BT`, `Saturation`(활력징후. 버튼으로 받을 수 없어 서빙 모델에 넣지 않음) |
| 뺌 | `Group`, `Patients number per hour`, `Arrival mode`(버튼으로 받을 수 없음) / `Diagnosis in ED`, `Disposition`, `Length of stay_min`, `KTAS duration_min`, `Error_group`, `mistriage`(내원 후 결과라 라벨 누출 위험) |

### 1.3 분할 [실제 진행: 03]

test 20%를 라벨로 층화해 먼저 뗀다(seed `20261027`). 남은 것에서 전체의 15%를 valid로 층화해 뗀다. id 목록은 `data/splits/`에 커밋한다. 재생성한 test 파일의 sha256이 커밋된 값과 다르면 평가 명령이 멈춘다.

In [ ]:
labels = splits_report["labels"]
pd.DataFrame(
    {
        name: {**{a: f"{labels[name]['acuity'][a]} / {labels[name]['total']}" for a in ACUITIES}, "합": labels[name]["total"]}
        for name in ("all", "train", "valid", "test")
    }
).T.rename(index={"all": "전체"})

세 분할 모두 응급 비율이 19.3–19.5%다. 층화가 유지됐다. 클래스별 건수가 가장 적은 곳은 **valid 응급 37건**이다. 이 숫자가 뒤의 임계값 선택을 불안정하게 만든다(6절, 7절).

## 2. 데이터 탐색

### 2.1 실제로 한 탐색 [실제 진행]

탐색은 02·03에서 했다. 04에서는 학습 전에 새로 탐색하지 않았다. 아래가 전부다.

**02(원본 확인, `data/README.md`에 기록)**
- 행·열 수, 열 이름이 Kaggle 판과 같은지 확인했다. Kaggle 판은 `Age`를 반올림한 사본이다
- 코드값을 확인했다(`Sex` 1=여 2=남, `Injury` 1=아니오 2=예, `Mental` 1–4 = AVPU)
- 라벨 분포: `KTAS_expert` 1=26, 2=220, 3=487, 4=459, 5=75
- 이상값과 결측: 활력징후에 문자열 `측불`(측정 불가)이 섞여 있다(`SBP` 25건 등). `NRS_pain` 빈칸 556건. `Age`에 소수(예 67.5)가 있다
- `Chief_complain`은 영어·한국어 자유 텍스트이고 고유값이 427개다
- 자유 텍스트에서 이메일·전화번호 형태를 찾아 식별 정보가 없음을 확인했다

**03(전처리하면서 확인, `backlog/03` 결과)**
- 주증상을 정규화(공백·대소문자)하면 고유값 422개가 남는다. 이를 버튼 어휘(`BodyPart`·`Symptom`)로 매핑할 수 있는지 한 줄씩 봤다 → 328개 매핑, 94개는 맞는 버튼이 없다
- 매핑 커버리지(행 기준): 전체 1,063/1,267(83.9%). 매핑 안 된 상위: `mental change` 22, `general weakness` 21, `syncope` 10, `dysarthria` 10, `post seizure` 7
- `Pain=0`인 553행은 `NRS_pain`이 비어 있다 → 통증 없음이므로 0점. `Pain=1`인데 점수가 빈 3행만 결측
- 발생 시점에 해당하는 열은 원본에 없다
- 분할별 라벨 건수와 층화 유지를 확인했다(1.3)

**04(학습 직전)**
- `data/processed/triage.parquet`의 열 목록과 첫 두 행으로 형식만 확인했다
- 피처별 분포, 클래스별 차이, 상관 같은 탐색은 **하지 않았다**. 03에서 정한 피처를 그대로 넣었다

### 2.2 train+valid 탐색 [사후 확인]

보고서를 쓰면서 train+valid 1,013행으로 계산했다. 모델 결정에는 쓰지 않았지만, 1라운드 개선 방향을 정하는 근거가 된다.

In [ ]:
dev = pd.concat([split.load_dev(DATA / "processed", DATA / "splits", name) for name in split.DEV_SPLITS], ignore_index=True)
X_dev = pd.DataFrame(featurize_many(dev.to_dict("records")), columns=FEATURE_NAMES)
print(f"train+valid {len(dev)}행, test 행 포함 여부: {dev['split'].isin(['test']).any()}")


def by_acuity(column, order=None):
    # 행 = 값, 열 = 응급도. 칸은 '응급도별 건수 / 그 값의 행 수'
    table = pd.crosstab(dev[column], dev["acuity"]).reindex(columns=list(ACUITIES), fill_value=0)
    if order is not None:
        table = table.reindex(order)
    total = table.sum(axis=1)
    out = table.astype(str).add(" / ").add(total.astype(str), axis=0)
    out["응급 비율"] = (table["emergency"] / total).map("{:.1%}".format)
    return out

**의식 상태(AVPU).** 의식이 명료하지 않은(verbal·pain·unresponsive) 59행 중 44행이 응급이다. 강한 신호지만 해당 행이 적다.

In [ ]:
by_acuity("mental", ["alert", "verbal", "pain", "unresponsive"])

**나이대.** 나이가 많을수록 응급 비율이 높다(65세 이상 101/368, 30세 미만 9/158).

In [ ]:
by_acuity("age_group", ["under30", "30to44", "45to64", "65plus"])

**손상 여부.** 손상(외상) 행은 대부분 외래·약국이다(139/193). 이 데이터의 응급은 주로 내과 계열이다.

In [ ]:
by_acuity("injury")

**통증 점수.** 응급 행의 통증 점수 평균이 1.86으로 세 클래스 중 가장 낮다(today 2.23, outpatient 2.55). 중앙값은 응급·today가 0, outpatient가 3이다. 응급 환자는 의식 저하 등으로 통증을 말하지 못하거나 통증이 없는 증상(의식 변화, 쇠약)으로 오는 경우가 많다. 그래서 "통증이 낮으면 덜 급하다"는 직관이 이 데이터에서는 맞지 않는다.

In [ ]:
dev.groupby("acuity")["pain_score"].describe().reindex(list(ACUITIES)).round(2)

**주증상 매핑과 응급.** 가장 중요한 사후 발견이다. 매핑이 안 된 행(부위·증상 피처가 전부 0)의 응급 비율이 **53/161(32.9%)**이다. 매핑된 행은 144/852(16.9%)다. 응급 환자의 주증상(의식 변화, 쇠약, 실신, 구음장애, 경련 후)이 지금 버튼 어휘에 없다. 그래서 응급 행일수록 증상 정보가 비어서 들어간다.

In [ ]:
display(by_acuity("chief_complaint_mapped"))
dev.loc[~dev["chief_complaint_mapped"], "chief_complaint"].value_counts().head(10).rename("매핑 안 된 행 수 (train+valid)").to_frame()

**부위·증상 피처의 클래스별 출현 비율**(많이 나온 순). 응급은 `body_chest`(42.6%)·`symptom_breath`·`symptom_cantmove`가 두드러진다. 외래는 `symptom_cut`·손발·피부 등 외상 계열이 많다.

In [ ]:
binary = [f for f in FEATURE_NAMES if f.startswith(("body_", "symptom_"))]
prevalence = X_dev[binary].groupby(dev["acuity"]).mean().T.reindex(columns=list(ACUITIES))
prevalence.insert(0, "행 수", X_dev[binary].sum().astype(int))
prevalence.sort_values("행 수", ascending=False).style.format({a: "{:.1%}" for a in ACUITIES})

**참고: 간호사 KTAS와 전문가 KTAS.** 원본의 간호사 판정(`KTAS_RN`)을 같은 3단계로 바꿔 전문가 라벨과 비교했다. 간호사는 활력징후와 대면 관찰을 쓰므로 버튼만 쓰는 모델과 조건이 다르다. 그래서 비교 기준이 아니라 참고로만 둔다.

In [ ]:
rn = pd.crosstab(dev["acuity"], dev["acuity_rn"]).reindex(index=list(ACUITIES), columns=list(ACUITIES), fill_value=0)
tp = rn.loc["emergency", "emergency"]
print(
    f"간호사 응급 재현율 {tp}/{rn.loc['emergency'].sum()}, 응급 정밀도 {tp}/{rn['emergency'].sum()}, 일치 {np.trace(rn.values)}/{rn.values.sum()}"
)
rn.rename_axis(index="전문가(라벨)", columns="간호사")

## 3. 전처리 [실제 진행: 03]

모든 전처리는 `src/fornav_modeling/data/kaggle_triage.py`(원본 → 표준 테이블)와 `src/fornav_modeling/features/buttons.py`(표준 테이블 행 → 피처 벡터)에 있다. 학습과 서빙이 같은 `featurize` 함수를 쓴다.

### 3.1 원본 → 표준 테이블

| 버튼 입력 | 원본 | 변환 규칙 | 결측 |
|---|---|---|---|
| `body_parts`, `symptoms` | `Chief_complain` | 정규화(앞뒤 공백 제거·소문자·공백 하나로) → `data/tables/chief_complaint_map.csv`(422행)로 부위·증상 목록. 표에 없는 주증상이 나오면 멈춘다 | 매핑 안 된 행은 빈 목록(전체 204행) |
| `pain_score` | `Pain`, `NRS_pain` | `Pain=0` → 0점, `Pain=1` → `NRS_pain` | `Pain=1`인데 점수 빈칸 3행 → NaN |
| `mental` | `Mental` | 1–4 → `alert`, `verbal`, `pain`, `unresponsive` | 없음 |
| `age_group` | `Age` | `<30`, `30–44`, `45–64`, `65+` 네 구간. 경계는 03에서 정함(앱에 나이대 버튼이 아직 없음) | 없음 |
| `sex` | `Sex` | 1 → `female`, 2 → `male` | 없음 |
| `injury` | `Injury` | 1 → False, 2 → True | 없음 |
| `onset` | (없음) | 원본에 발생 시점 열이 없다 | **전체 NaN** |
| 라벨 `acuity` | `KTAS_expert` | 1–2 → `emergency`, 3 → `today`, 4–5 → `outpatient`(`labels/ktas.py`) | 없음 |

코드값이 정의 밖이면 조용히 넘기지 않고 `ValueError`로 멈춘다.

**주증상 매핑 표.** 키워드 규칙으로 초안을 만든 뒤 Claude(LLM)가 422행을 한 줄씩 고쳤다. 출처 열에 그렇게 적었고 전부 `reviewed=no`다. **사람 검토 전**이다. 판단이 들어간 규칙은 `note` 열에 남겼다(예: 어지러움 → `head`, 옆구리 → `back`, 호흡곤란 → `chest`). 이 표는 test 행의 주증상 텍스트까지 포함한 422개 전체로 만들었다. 라벨을 보지 않는 어휘 대응이지만, 엄밀히 말하면 test 쪽 텍스트를 본 셈이다.

### 3.2 표준 테이블 → 피처 벡터(34개)

| 피처 | 개수 | 인코딩 |
|---|---|---|
| `body_*` | 13 | 여러 개 선택 → 0/1 |
| `symptom_*` | 15 | 여러 개 선택 → 0/1 |
| `pain_score` | 1 | 0–10 그대로 |
| `mental` | 1 | AVPU 순서 0–3 |
| `age_group` | 1 | 0–3 |
| `onset` | 1 | 0–3(학습 데이터는 전부 NaN) |
| `sex_male` | 1 | 0/1 |
| `injury` | 1 | 0/1 |

결측은 NaN으로 둔다. LightGBM은 결측을 그대로 다룬다. 로지스틱 회귀는 train 중앙값으로 채운다(5절). 스케일링·원-핫은 LightGBM에 필요 없어 하지 않았다. 순서형 값(AVPU·나이대)은 순서 정수로 두었다.

예시(train 첫 행):

In [ ]:
row = dev.iloc[0]
print(
    {
        k: (list(v) if isinstance(v, np.ndarray) else v)
        for k, v in row[
            ["id", "chief_complaint", "body_parts", "symptoms", "pain_score", "mental", "age_group", "onset", "sex", "injury", "acuity"]
        ].items()
    }
)
pd.Series(X_dev.iloc[0].values, index=FEATURE_NAMES).loc[lambda s: s != 0].rename("0이 아닌 피처 값").to_frame()

### 3.3 피처 결측 [실제 진행: 03]

`onset`은 train+valid 전체가 결측이라 모델이 쓸 수 없다. 서빙 때 값이 들어와도 예측이 바뀌지 않는지 04에서 확인했다(5.3).

In [ ]:
X_dev.isna().sum().loc[lambda s: s > 0].rename("결측 행 수 (train+valid 1,013행 중)").to_frame()

## 4. 학습·평가 절차 [실제 진행: 04]

```
train 823행 ──▶ 라운드 수: train 안 층화 5겹 CV 조기 종료 ──▶ LightGBM 학습(86라운드)
           └──▶ 로지스틱 회귀 학습
valid 190행 ──▶ 두 모델 각각 임계값 선택(응급 재현율 ≥ 95% 중 정밀도 최대) ──▶ valid 지표
train+valid ──▶ 5겹 교차검증 예측(라운드 수·임계값 고정) ──▶ CV 지표(참고)
test 254행  ──▶ 저장한 모델 + valid 임계값으로 1회 평가 ──▶ 최종
```

- 명령: `uv run python scripts/train_acuity.py`(약 1초), `uv run python scripts/evaluate.py --model acuity --round v0`(test 1회. 두 번째 실행은 거부된다)
- valid는 모델 학습(가중치·라운드 수)에 쓰지 않았다. 임계값 선택과 평가에만 썼다
- 서빙 후보(LightGBM vs 로지스틱)는 valid와 CV 수치로 정했고, 그 뒤에 test를 봤다
- 같은 seed로 두 번 학습하면 `model.txt`의 sha256이 같다(`tests/test_determinism.py`, 실제 데이터로도 확인)

## 5. 하이퍼파라미터

### 5.1 어떻게 정했나 [실제 진행]

**그리드·랜덤 탐색은 하지 않았다.** 데이터가 train 823행뿐이다. 그래서 과적합을 막는 쪽의 보수적인 값을 처음에 한 번 정하고 바꾸지 않았다. 데이터로 정한 것은 **부스팅 라운드 수 하나**다. 탐색하지 않은 이유는 두 가지다. ① valid 응급 37건으로 여러 설정을 비교하면 우연히 좋은 설정을 고르기 쉽다. ② v0의 목적이 기준선이다. 탐색을 하려면 train 안 교차검증으로 하는 것이 맞고, 1라운드 후보로 남긴다.

In [ ]:
reasons = {
    "objective": ("multiclass", "3단계 확률을 한 번에 낸다"),
    "num_class": (3, "emergency / today / outpatient"),
    "learning_rate": (0.05, "작은 데이터에서 흔히 쓰는 값. 라운드 수를 CV로 맞추므로 크게 민감하지 않다고 봄(검증 안 함)"),
    "num_leaves": (15, "기본값 31의 절반. 823행에 비해 트리가 너무 깊지 않게"),
    "min_data_in_leaf": (20, "기본값. 잎 하나에 최소 20행 → 희귀 조합 암기 방지"),
    "lambda_l2": (1.0, "약한 L2 규제. 기본값 0에서 올림"),
    "num_boost_round": (metadata["num_rounds"], "train 안 층화 5겹 CV, multi_logloss가 50라운드 동안 좋아지지 않으면 멈춤 → 최저점 라운드"),
    "seed / deterministic / num_threads / force_row_wise": ("20261028 / True / 1 / True", "같은 seed 두 번 → 같은 모델 문자열"),
    "클래스 가중치": ("없음", "불균형(응급 19%)은 임계값으로 다룬다"),
}
assert all(LGBM_PARAMS[k] == v for k, (v, _) in reasons.items() if k in LGBM_PARAMS)  # 표가 코드와 같은지
pd.DataFrame(reasons, index=["값", "정한 방법·이유"]).T

### 5.2 라운드 수: train 안 교차검증 곡선 [사후 확인으로 다시 그림]

실제 학습은 `lgb.cv`의 조기 종료 결과(86)만 썼다. 아래는 같은 폴드로 조기 종료 없이 300라운드까지 그려 86이 어디인지 보인다.

In [ ]:
train = split.load_dev(DATA / "processed", DATA / "splits", "train")
X_train, y_train = featurize_many(train.to_dict("records")), encode_labels(train["acuity"])

curve = lgb.cv(
    LGBM_PARAMS,
    lgb.Dataset(X_train, label=y_train, feature_name=list(FEATURE_NAMES)),
    num_boost_round=300,
    folds=StratifiedKFold(acuity.CV_FOLDS, shuffle=True, random_state=acuity.SEED).split(X_train, y_train),
)
mean, std = np.array(curve["valid multi_logloss-mean"]), np.array(curve["valid multi_logloss-stdv"])
best = int(mean.argmin()) + 1
assert best == metadata["num_rounds"], (best, metadata["num_rounds"])
rounds = np.arange(1, len(mean) + 1)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.fill_between(rounds, mean - std, mean + std, color=BLUE, alpha=0.12, linewidth=0)
ax.plot(rounds, mean, color=BLUE)
ax.scatter([best], [mean[best - 1]], s=40, color=BLUE, edgecolor=SURFACE, linewidth=2, zorder=3)
ax.annotate(
    f"최저 {best}라운드, logloss {mean[best - 1]:.3f}",
    (best, mean[best - 1]),
    xytext=(12, -22),
    textcoords="offset points",
    color=INK,
    fontsize=9,
)
ax.set(xlabel="부스팅 라운드", ylabel="검증 폴드 multi_logloss (평균 ± 표준편차)", title="train 823행, 층화 5겹 교차검증")
plt.tight_layout()

### 5.3 비교 모델과 발생 시점 확인 [실제 진행]

- **로지스틱 회귀**: 결측은 train 중앙값으로 채우고(전체 결측인 `onset`은 0으로 남겨 피처 순서 유지), 표준화한 뒤 다항 로지스틱(`C=1.0`, 기본값, 탐색 안 함)을 쓴다. 학습 후 수치(중앙값·평균·표준편차·계수)를 JSON으로 저장하고 numpy로 예측한다(pickle 미사용)
- **발생 시점(onset)**: valid 190행의 `onset`을 0–3으로 채워 넣어도 두 모델의 확률 변화 최댓값이 0이다. 학습에서 본 적 없는 피처라 서빙 때 값이 들어와도 예측에 영향이 없다

In [ ]:
{name: r["onset_max_proba_change"] for name, r in valid_report["models"].items()}

## 6. 임계값 [실제 진행]

**규칙.** 결정 규칙은 다음과 같다. `emergency` 확률 ≥ 임계값이면 응급, 아니면 `today`·`outpatient` 중 확률이 큰 쪽.

임계값은 valid에서 이렇게 고른다.
1. 응급 재현율 목표를 먼저 정한다. **95%**(사용자와 정함)
2. 목표를 만족하는 임계값 중 응급 정밀도가 가장 높은 것을 고른다
3. 경계는 고른 행의 확률과 바로 아래 확률의 중간에 둔다. 저장·불러오기 때 소수점 차이로 판정이 뒤집히지 않게 하려는 것이다

valid 응급이 37건이므로 95%는 36건 이상을 뜻한다. 1건만 놓쳐도 된다.

In [ ]:
valid = split.load_dev(DATA / "processed", DATA / "splits", "valid")
X_valid, y_valid = featurize_many(valid.to_dict("records")), encode_labels(valid["acuity"])
model = LightGBMAcuity.from_text((MODEL_DIR / acuity.MODEL_FILE).read_text())
p_em = model.predict_proba(X_valid)[:, EMERGENCY]
chosen = metadata["thresholds"]["lightgbm"]

ts = np.unique(np.concatenate([p_em, [chosen["threshold"]]]))
positive = y_valid == EMERGENCY
recall = np.array([(positive & (p_em >= t)).sum() / positive.sum() for t in ts])
precision = np.array([(positive & (p_em >= t)).sum() / max((p_em >= t).sum(), 1) for t in ts])
flagged_share = np.array([(p_em >= t).mean() for t in ts])

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.step(ts, recall, where="post", color=BLUE, label="응급 재현율")
ax.step(ts, precision, where="post", color=ORANGE, label="응급 정밀도")
ax.axhline(0.95, color=MUTED, linewidth=1, linestyle=(0, (4, 3)))
ax.axvline(chosen["threshold"], color=INK, linewidth=1)
ax.text(
    chosen["threshold"] + 0.01,
    0.06,
    f"선택 {chosen['threshold']:.4f}\n재현율 {chosen['recall'][0]}/{chosen['recall'][1]}, 정밀도 {chosen['precision'][0]}/{chosen['precision'][1]}",
    fontsize=9,
)
ax.text(0.01, 0.935, "목표 95%", fontsize=9, color=MUTED, ha="left", va="top", transform=ax.get_yaxis_transform())
ax.set_xscale("log")
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.set(xlabel="emergency 확률 임계값 (로그 축)", ylabel="비율", ylim=(0, 1.04), title="valid 190행(응급 37건) LightGBM")
ax.legend(loc="center left", frameon=False)
plt.tight_layout()

assert chosen["recall"] == [int((positive & (p_em >= chosen["threshold"])).sum()), int(positive.sum())]  # 불러온 모델로 다시 계산해도 같다
print(f"이 임계값에서 valid {int((p_em >= chosen['threshold']).sum())}/{len(p_em)}행을 응급으로 본다")

재현율 95%를 맞추려면 임계값을 0.074까지 내려야 한다. 그러면 valid의 절반(99/190)을 응급으로 본다. 그래프에서 재현율은 임계값 0.07–0.2 구간에서 한 건씩 계단식으로 떨어진다. 응급 37건 중 마지막 몇 건이 낮은 확률에 흩어져 있어서다. 이 몇 건이 임계값을 정한다. 그래서 다른 표본(test)에서는 같은 임계값이 같은 재현율을 보장하지 않는다(7절).

## 7. 결과

### 7.1 valid와 train+valid 교차검증 [실제 진행]

칸은 `분자/분모 (비율)`이다. `argmax`는 임계값 없이 확률이 가장 큰 클래스, `threshold`는 6절의 규칙이다. CV는 train+valid를 층화 5겹으로 나눈 예측이다. 라운드 수 86과 valid로 정한 임계값을 그대로 썼다. valid 행에 대해서는 임계값이 그 행으로 정해졌으므로 낙관적이다.

In [ ]:
def cell(r):
    return f"{r['n']}/{r['of']} ({r['value']:.1%})" if r["value"] is not None else f"{r['n']}/{r['of']}"


def metrics_frame(results):
    rows = {}
    for name, m in results.items():
        rows[name] = {
            **{f"{a} 재현율": cell(m["per_class"][a]["recall"]) for a in ACUITIES},
            **{f"{a} 정밀도": cell(m["per_class"][a]["precision"]) for a in ACUITIES},
            "정확도": cell(m["accuracy"]),
        }
    return pd.DataFrame(rows).T


metrics_frame(
    {
        f"{n} {part} ({rule})": r[part][rule]
        for part in ("valid", "cv_train_valid")
        for n, r in valid_report["models"].items()
        for rule in r[part]
    }
)

- 같은 응급 재현율(36/37)에서 LightGBM의 응급 정밀도(36/99)가 로지스틱(36/112)보다 높다. CV에서도 LightGBM이 재현율·정밀도·정확도 모두 높다 → 서빙 후보는 LightGBM
- 임계값을 적용하면 응급 재현율은 오르고, `today` 재현율은 크게 떨어진다(valid 47/73 → 26/73). `today` 환자 상당수가 응급으로 넘어간다
- CV 응급 재현율(184/197, 93.4%)이 valid(36/37)보다 낮다. valid 37건으로 정한 임계값이 다른 행에서는 목표보다 낮게 나올 수 있다는 첫 신호였다

### 7.2 최종 test (1회) [실제 진행]

In [ ]:
print(
    f"final={test_report['final']}, rows={test_report['rows']}, 모델 파일 해시가 metadata와 같음: {test_report['files'] == metadata['files']}"
)
metrics_frame({f"{n} ({rule})": r[rule] for n, r in test_report["models"].items() for rule in ("argmax", "threshold_rule")})

In [ ]:
def confusion(m):
    c = m["confusion"]
    return pd.DataFrame(c["matrix"], index=[f"정답 {a}" for a in c["labels"]], columns=[f"예측 {a}" for a in c["labels"]])


print("valid, LightGBM 임계값")
display(confusion(valid_report["models"]["lightgbm"]["valid"]["threshold"]))
print("test, LightGBM 임계값 (최종)")
display(confusion(test_report["models"]["lightgbm"]["threshold_rule"]))

- **test 응급 재현율 42/49(85.7%)**. 목표 95%에 못 미치고 7건을 놓친다. 놓친 7건 중 4건은 `today`, 3건은 `outpatient`로 갔다
- 응급 판정 129건 중 87건(정답 today 53, outpatient 34)은 응급이 아니다
- 참고로 train+valid에서 간호사 KTAS의 응급 재현율은 171/197(86.8%)이었다(2.2). 간호사는 활력징후와 대면 관찰을 쓰므로 직접 비교할 수는 없다. 다만 버튼만 쓰는 모델이 이 수준의 재현율을 내려면 정밀도를 크게 내줘야 한다
- 놓친 7건은 test라 열어 보지 않았다. 원인 분석은 train·valid 오류로 한다

## 8. SHAP 판단 근거

### 8.1 방법 [실제 진행]

LightGBM 내장 TreeSHAP(`pred_contrib=True`)으로 행마다 클래스별 기여도를 구한다. `shap.TreeExplainer`와 값이 같은지 테스트로 확인했다. 기여도 단위는 softmax 전 원점수(log-odds)다. 양수면 그 클래스 쪽으로 민다. 리포트에는 최종 예측 클래스에 대한 |기여도| 상위 3개를 `{feature, value, contribution}`으로 낸다.

### 8.2 예시 3건 [실제 진행]

valid에서 최종 예측 클래스마다 id 순 첫 행이다.

In [ ]:
for ex in valid_report["shap_examples_valid"]:
    print(f"{ex['id']}  정답 {ex['true']} → 예측 {ex['predicted']}  확률 {ex['proba']}")
    for t in ex["top"]:
        print(f"    {t['feature']:<16} 값 {t['value']}  기여 {t['contribution']:+.3f}")

첫 예시(`triage_0006`)는 응급 확률 0.075로 임계값 바로 위에 걸려 응급으로 판정됐다. 정답은 외래다. 이렇게 낮은 임계값으로 응급이 된 행은 응급 쪽 기여가 작다. SHAP 설명을 사용자 문장에 넣을 때 "이 증상 때문에 응급"이라고 단정하지 않도록 해야 한다.

### 8.3 응급 클래스 전역 중요도 [사후 확인]

valid 190행의 응급 클래스 |SHAP| 평균이다.

In [ ]:
sv = explain.shap_values(model.booster, X_valid)[:, EMERGENCY]
importance = pd.Series(np.abs(sv).mean(axis=0), index=FEATURE_NAMES).sort_values(ascending=False)
top = importance.head(12)[::-1]

fig, ax = plt.subplots(figsize=(7.0, 4.2))
ax.barh(top.index, top.values, color=BLUE, height=0.6)
for y, v in enumerate(top.values):
    ax.text(v + 0.005, y, f"{v:.3f}", va="center", fontsize=8, color=MUTED)
ax.set(xlabel="응급 클래스 |SHAP| 평균 (원점수)", title="valid 190행, LightGBM 상위 12개 피처")
ax.grid(axis="y", visible=False)
plt.tight_layout()
print("기여가 0인 피처:", list(importance[importance == 0].index))

응급 판단에는 `body_chest`, `age_group`, `pain_score`, `mental`이 가장 크게 쓰인다. 2.2의 탐색과 맞는다(흉부 42.6%, 고령, 의식 저하). `pain_score`는 낮을수록 응급 쪽으로 미는 경우가 많다. valid에서 기여가 0인 피처(손·귀코목·치아·팔다리 등 외상 계열과 `onset`)도 많다. 이 데이터에서 응급과 관련이 적거나 행이 적어 트리가 쓰지 않은 것이다.

## 9. 한계와 다음 단계

**이 결과의 한계**
1. **test 응급 재현율 85.7%가 목표 95%에 못 미친다.** 임계값을 valid 응급 37건으로 정해서 표본 변동에 취약했다
2. **응급 정밀도가 낮다**(test 42/129). 실제 서비스에서는 응급이 아닌 사람을 응급실로 많이 안내하게 된다. 모델 앞의 레드플래그 규칙과 함께 쓸 때의 성능은 아직 재지 않았다
3. **주증상 매핑 표가 사람 검토 전**이다. 응급 환자의 주증상(의식 변화·쇠약·실신·경련)이 버튼 어휘에 없다. 그래서 응급 행일수록 증상 피처가 비어서 들어간다(2.2, 응급 비율 32.9% vs 16.9%)
4. **하이퍼파라미터를 탐색하지 않았다.** 라운드 수 외에는 정한 값의 민감도를 확인하지 않았다
5. 데이터가 국내 응급실 2곳의 내원 환자다. 외국인 근로자가 앱에 입력하는 상황(응급실에 오기 전, 자가 보고)과 분포가 다를 수 있다
6. `onset`은 학습 데이터에 없어 모델이 쓰지 못한다

**1라운드 후보**(사용자와 정한 뒤, test는 다시 보지 않고 새 라운드로 기록)
- 매핑 표 사람 검토, 의식 변화·쇠약·실신 같은 버튼 어휘 추가(백엔드 `vocab.py`와 함께 정해야 함)
- 임계값을 valid 37건 대신 train+valid 교차검증 예측(응급 197건)으로 정하기
- train 안 교차검증으로 `num_leaves`·`min_data_in_leaf`·`lambda_l2`의 작은 탐색, 클래스 가중치 비교
- 레드플래그 규칙을 앞에 둔 전체 흐름 기준의 응급 재현율 측정